## TF-IDF

**Term frequency** ($TF(i, d)$) is the number of occurrences of word (token) $i$ in document $d$. It depends strongly
on how general a word is (e.g. "has" vs. "cosine" in general literature) and also on the length of the document. Because of the length dependence,
it is frequently expressed as a share of all words in a document.

**Document frequency** ($DF(i)$) is the number of documents that contain word $i$.

**Inverse document frequency** ($IDF(i)$) is simply the inverse relative frequency of the word in the set of documents.
With $N$ documents the IDF is given by:

$$
    DF(i) = \frac{DF(i)}{N}
$$

$$
    IDF(i) = \frac{N}{DF(i)}
$$

It is large for words that occur in many documents, and it will be small for words that appear in only a few documents.

**Smoothed TF-IDF**

A problem with this definition is that the IDF becomes very large for large corpora (large N) so it is commonly replaced
by its logarithm.

$$
\text{IDF}(i) = 1 + \log\left(\frac{N}{DF(i)}\right)
$$

The addition of 1 in the above equation serves to ensure that the words that occur in all documents are not entirely discarded. The default IDF used in `TfidfVectorizer` is:

$$
\text{IDF}(i) = 1 + \log\left(\frac{N + 1}{DF(i) + 1}\right)
$$

$$
\text{TF-IDF}(i, d) = TF(i, d) \times \text{IDF}(i)
$$

Let's calculate it for the toy corpus with just three documents:

```
    "the quick brown fox",
    "the fast brown dog",
    "the quick red fox"
```

In [ ]:
import math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

corpus = [
    'the small quick brown fox',
    'the small fast brown dog',
    'the small quick red fox'
]

c_vect = CountVectorizer()

term_matrix = c_vect.fit_transform(corpus)
term_matrix_dense = term_matrix.toarray()

pd.DataFrame(term_matrix_dense, columns=c_vect.get_feature_names_out())

In [ ]:
tfidf_vect = TfidfVectorizer(smooth_idf=True, use_idf=True, norm=None)
tfidf_term_matrix = tfidf_vect.fit_transform(corpus)
pd.DataFrame(
    tfidf_term_matrix.toarray(),
    columns=c_vect.get_feature_names_out(),
    index=[f"doc{i}" for i in range(1, len(corpus) + 1)]
    )

In [ ]:
# Get the inverse document frequency
tfidf_vect.idf_

In [ ]:
# Read in the data
df = pd.read_csv("https://github.com/febse/data/raw/refs/heads/main/ta/reviews.csv")
df.head()

In [ ]:
# Define the vectorizer

vectorizer = TfidfVectorizer(    
    strip_accents="unicode",
    lowercase=True, # Default is True
    # stop_words=list(sw),
    # ngram_range=(1, 2),
    min_df=5,
    max_df=0.95
)

# Remove missing values
df.dropna(inplace=True)

# Drop reviews with neutral ratings
df = df[df['Rating'] != 3]

df["positive"] = np.where(df['Rating'] > 3, 1, 0)

# Create bag-of-words features using TF

X_train, X_test, y_train, y_test = train_test_split(df['Reviews'],
                                                    df['positive'],
                                                    random_state=0)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Number of features: {X_train_vec.shape[1]}')
print(f'Training set shape: {X_train_vec.shape}')
print(f'Test set shape: {X_test_vec.shape}')

In [ ]:
# Train logistic regression classifier
logreg = LogisticRegression(max_iter=1000, random_state=0)
logreg.fit(X_train_vec, y_train)

# Make predictions
y_pred = logreg.predict(X_test_vec)
y_pred_proba = logreg.predict_proba(X_test_vec)[:, 1]

# Calculate accuracy and AUC
accuracy = logreg.score(X_test_vec, y_test)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'AUC Score: {auc_score:.4f}')

In [ ]:
example = "I'm not happy"

example_vec = vectorizer.transform([example])

example_vec.toarray()

example_pred_proba = logreg.predict_proba(example_vec)[:, 1]
print(f'Predicted probability of positive review for example "{example}": {example_pred_proba[0]:.4f}')

In [ ]:
# Get feature names and coefficients
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = logreg.coef_[0]

# Create a dataframe with features and their coefficients
feature_coef = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

# Top 20 features associated with positive sentiment
print("Top 20 features most associated with POSITIVE sentiment:")
print(feature_coef.head(20).to_string(index=False))

print("\n" + "="*60 + "\n")

# Top 20 features associated with negative sentiment
print("Top 20 features most associated with NEGATIVE sentiment:")
print(feature_coef.tail(20).to_string(index=False))